# 01 - Contrato, EDA e baseline

Parte da Pessoa 1. Este notebook valida os arquivos oficiais, registra riscos de vazamento e cria um baseline reproduzível com Árvore de Decisão. Nenhuma decisão é tomada com o conjunto de teste.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.baseline import evaluate_baseline
from src.data_validation import (
    CATEGORICAL_COLUMNS, DatasetPaths, load_competition_data,
    quality_report, validate_contract,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

## 1. Carga e contrato

Se esta célula falhar, confira se os três CSVs do Canvas estão em `data/` com os nomes originais.

In [ ]:
paths = DatasetPaths.from_directory(PROJECT_ROOT / 'data')
train, test, submission_model = load_competition_data(paths)
errors = validate_contract(train, test, submission_model)
assert not errors, '\n'.join(errors)
quality_report(train, test)

## 2. Distribuição do alvo e qualidade dos atributos

Duplicatas não são removidas automaticamente: primeiro é preciso distinguir duplicatas exatas, atributos repetidos e observações legítimas com os mesmos valores.

In [ ]:
display(train.head())
display(train['class'].value_counts().rename('quantidade').to_frame())
display((train['class'].value_counts(normalize=True) * 100).rename('percentual').to_frame())

ax = sns.countplot(data=train, x='class', order=train['class'].value_counts().index)
ax.set(title='Distribuição das classes', xlabel='Classe', ylabel='Instâncias')
plt.show()

In [ ]:
categorical_summary = pd.DataFrame({
    'cardinalidade_treino': train[list(CATEGORICAL_COLUMNS)].nunique(),
    'cardinalidade_teste': test[list(CATEGORICAL_COLUMNS)].nunique(),
    'ausentes_treino': train[list(CATEGORICAL_COLUMNS)].isna().sum(),
    'ausentes_teste': test[list(CATEGORICAL_COLUMNS)].isna().sum(),
})
categorical_summary

## 3. Auditoria inicial de possível vazamento

A correlação abaixo serve para investigação, não para remoção automática. Correlação alta pode representar um sinal legítimo; vazamento depende de como o atributo foi produzido e se ele existe no momento da previsão. `id`, alvo e eventuais colunas de dificuldade devem ser analisados explicitamente.

In [ ]:
numeric = train.drop(columns=['class', 'id'], errors='ignore').select_dtypes(include='number')
target_binary = train['class'].str.lower().map({'normal': 0, 'anomaly': 1})
numeric_target_correlation = (
    numeric.corrwith(target_binary).abs().sort_values(ascending=False).rename('correlacao_abs')
)
display(numeric_target_correlation.to_frame().head(15))

suspicious_tokens = ('class', 'label', 'target', 'attack', 'difficulty', 'score', 'prediction')
suspicious_names = [
    column for column in train.columns
    if column != 'class' and any(token in column.lower() for token in suspicious_tokens)
]
print('Nomes que exigem justificativa:', suspicious_names)

## 4. Baseline sem balanceamento

A imputação e o one-hot encoding são ajustados dentro de cada dobra. Usamos `StratifiedKFold` sem embaralhamento para seguir a interpretação conservadora do enunciado.

In [ ]:
fold_results, class_report = evaluate_baseline(train, n_splits=5)
display(fold_results)
display(fold_results.describe().loc[['mean', 'std']])
display(pd.DataFrame(class_report).T)

## 5. Registro para integração

Antes de encerrar esta parte, registrar no relatório: tamanho dos conjuntos; proporção das classes; ausentes, duplicatas e constantes; cardinalidade de `service`; atributos suspeitos e decisão fundamentada; média e desvio-padrão do F1-macro do baseline. Não remover atributos apenas por correlação.